<a href="https://colab.research.google.com/github/sevaradevoloper/machine-learning-class-work/blob/main/activation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# activation formulalari

In [3]:
import numpy as np
z = np.array([-4., -2. , 0. , 2. , 4.])

sig = 1/ (1+ np.exp(-z))
print(sig.round(4))
print(sig * (1-sig).round(4))
print(np.tanh(z).round(4))


[0.018  0.1192 0.5    0.8808 0.982 ]
[0.01766246 0.10499393 0.25       0.10499101 0.01767625]
[-0.9993 -0.964   0.      0.964   0.9993]


In [4]:
z = np.array([-4. , -2. , 2. , 4.])
print(np.maximum(0,z))
print(np.where(z>0,z ,0.01 *z))
print(np.where(z>0,1.0, 0.01))

[0. 0. 2. 4.]
[-0.04 -0.02  2.    4.  ]
[0.01 0.01 1.   1.  ]


In [8]:
import torch
import torch.nn.functional as F

for f in [F.relu, F.leaky_relu]:
    w = torch.tensor(1.0, requires_grad=True)
    h = f(w * 2 - 5)
    ((h - 1)**2).backward()
    # w.grad uchun .item() to'g'ri yozildi, h esa tensor ko'rinishida chiqariladi
    print(h.detach().numpy().round(4), round(w.grad.item(), 4))


0.0 0.0
-0.03 -0.0412


In [11]:
# MSE katta xatolarni juda qattiq jazolaydi.
# MAE esa barcha xatolarga teng koʻz bilan qaraydi.
x = np.array([1., 2., 3., 4., 5.])
y = np.array([2., 4., 6., 8., 10.])

for a , b in ((2,0),(8,-12)):
  p = a*x+b
  print(f"W:{a}, b:{b}")
  print(f"MSE: {np.mean(y-p)**2}")
  print(f"MAE: {np.mean(np.abs(y-p))}" )

W:2, b:0
MSE: 0.0
MAE: 0.0
W:8, b:-12
MSE: 36.0
MAE: 8.4


In [12]:
# softmax

def softmax(z):
  e = np.exp(z - np.max(z))
  return e/e.sum()
print(softmax(np.array([2. ,1. , 0.])).round(4))
print(softmax(np.array([1000. , 1001.])).round(4))

[0.6652 0.2447 0.09  ]
[0.2689 0.7311]


In [14]:
# wine loyhasida barcha aktivatsiyalarni qilib chiqamiz:

import torch, torch.nn as nn , torch.nn.functional as F
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X,y  = load_wine(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42,stratify=y)

scaler = StandardScaler().fit(X_train)
A,B =  [torch.tensor(scaler.transform(v)).float() for v in (X_train, X_test)]
ya,yb = torch.tensor(y_train), torch.tensor(y_test)
print(f"X shape: { X.shape}, A.shape: {tuple(A.shape)}, B.shape: {tuple(B.shape)}")


''

X shape: (178, 13), A.shape: (124, 13), B.shape: (54, 13)


In [18]:
ACTS = {"sigmoid": nn.Sigmoid,
        "tanh:": nn.Tanh,
        "relu": nn.ReLU,
        "leaky": nn.LeakyReLU}
def fit_eval(act, lr = 0.05 , seed = 42, loss_fn = F.cross_entropy):
  torch.manual_seed(seed),
  a = ACTS[act]
  net = nn.Sequential(nn.Linear(13,16), a(), nn.Linear(16,16), a(),
                      nn.Linear(16,16), a(),nn.Linear(16,3)
                      )

  opt = torch.optim.SGD(net.parameters(),lr = lr)
  for _ in range(100):
     loss = loss_fn(net(A),ya)
     opt.zero_grad()
     loss.backward()
     opt.step()
  return round((net(B).argmax(1) == yb).float().mean().item(),4)

for lr in(0.05 , 0.01):
  for act in ACTS:
    print(lr,act,fit_eval(act,lr))

0.05 sigmoid 0.3889
0.05 tanh: 0.9815
0.05 relu 0.9444
0.05 leaky 0.963
0.01 sigmoid 0.3889
0.01 tanh: 0.7222
0.01 relu 0.3889
0.01 leaky 0.3889


In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def sigmoid(z):
    return 1 / (1 + np.exp(-z))


def d_sigmoid(z):
    s = sigmoid(z)
    return s * (1 - s)


def tanh(z):
    exp_z = np.exp(z)
    exp_minus_z = np.exp(-z)
    return (exp_z - exp_minus_z) / (exp_z + exp_minus_z)


def d_tanh(z):
    t = tanh(z)
    return 1 - t**2


def relu(z):
    return np.maximum(0, z)


def d_relu(z):
    return (z > 0).astype(float)


def leaky_relu(z, alpha=0.01):
    return np.where(z > 0, z, alpha * z)


def d_leaky_relu(z, alpha=0.01):
    return np.where(z > 0, 1.0, alpha)


def softmax(z):
    shift_z = z - np.max(z, axis=-1, keepdims=True)
    exp_z = np.exp(shift_z)
    return exp_z / np.sum(exp_z, axis=-1, keepdims=True)


ACT = {
    "sigmoid": (sigmoid, d_sigmoid),
    "tanh": (tanh, d_tanh),
    "relu": (relu, d_relu),
    "leaky_relu": (leaky_relu, d_leaky_relu),
}
